# Setup

In [3]:
import numpy as np

from qiskit import QuantumCircuit, transpile
from qiskit.circuit.library import HGate, XGate, SGate, SdgGate
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, depolarizing_error, ReadoutError
from qiskit.result import marginal_counts
from qiskit.visualization import plot_histogram

In [4]:
# Magic numbers
OPT_LEVEL = 0
SHOTS = 4096
SEED = 7
LAM1, LAM2 = 0.005, 0.02
P = 0.03

# default collections
NOISE = ("ideal", "gates", "readout", "combined")
BASIS_1Q = ['h','x','z','s','sdg','ry','rz','id']
BASIS_2Q = ['cx']
BASIS = BASIS_1Q + BASIS_2Q

In [5]:
# helper functions
def add_noise(kind: str):
    # add noise to the AerSimulator
    nm = NoiseModel()
    if kind in ("gates", "combined"):
        nm.add_all_qubit_quantum_error(depolarizing_error(LAM1, 1), BASIS_1Q)
        nm.add_all_qubit_quantum_error(depolarizing_error(LAM2, 2), BASIS_2Q)
    if kind in ("readout", "combined"):
        nm.add_all_qubit_readout_error(ReadoutError([[1 - P, P],[P, 1 - P]]))
    return nm

def run(qc: QuantumCircuit, 
        noise: str, 
        shots: int = SHOTS, 
        seed: int = SEED, 
        opt_level: int = OPT_LEVEL):
    # run the thing
    sim = AerSimulator(method="density_matrix",
                       noise_model= add_noise(noise),
                       basis_gates=BASIS)
    tqc = transpile(qc, sim, optimization_level=opt_level, seed_transpiler=seed)
    result = sim.run(tqc, shots=shots, seed_simulator=seed).result()
    return result, tqc

# Part A: Quantum Teleportation

In [ ]:
PROBES = {
    "0": [],
    "1": [XGate()],
    "+": [HGate()],
    "-": [XGate(), HGate()],
    "+i": [HGate(), SGate()],
    "-i": [HGate(), SdgGate()]
}

# get the preparation and inverse-preparation stuff
def get_prep_circuits(probe):
    prep = QuantumCircuit(1, name="prep")
    for gate in PROBES[probe]:
        prep.append(gate, [0])
    post = prep.inverse()
    post.name = "post"
    return prep.to_gate(), post.to_gate()

def quantum_teleportation(probe, bob_corrects=True):
    # q0 = unknown state, q1 = alice qubit, q2 = bob qubit
    # bob_corrects is whether bob will correct his qubit
    q = QuantumCircuit(3, 3)

    # prepare qubit
    prep, post = get_prep_circuits(probe)
    q.append(prep, [0])
    q.barrier()

    # alice and bob share e-pair
    q.h(1)
    q.cx(1, 2)
    q.barrier()
    
    # start quantum tele
    q.cx(0, 1)

    # alice measure
    q.h(0)
    q.measure(0, 0)
    q.measure(1, 1)

    # bob corrects
    if bob_corrects:
        with q.if_test((q.clbits[0], 1)):
            q.z(2)
        with q.if_test((q.clbits[1], 1)):
            q.x(2)
    q.barrier()

    # post-processing + measure
    q.append(post, [2])
    q.measure(2, 2)
    return q

## Circuit
Using an arbitrary 0 starting state and assuming Bob corrects as usual:
- `prep_probe` prepares the probe
- `post_probe` post-processes Bob's qubit (with the inverse preparation operation)

In [42]:
quantum_teleportation("0").draw()

┌──────┐ ░            ░      ┌───┐┌─┐                                     »
q_0: ┤ prep ├─░────────────░───■──┤ H ├┤M├─────────────────────────────────────»
     └──────┘ ░ ┌───┐      ░ ┌─┴─┐└┬─┬┘└╥┘                                     »
q_1: ─────────░─┤ H ├──■───░─┤ X ├─┤M├──╫──────────────────────────────────────»
              ░ └───┘┌─┴─┐ ░ └───┘ └╥┘  ║   ┌──────  ┌───┐ ───────┐   ┌──────  »
q_2: ─────────░──────┤ X ├─░────────╫───╫───┤ If-0  ─┤ Z ├  End-0 ├───┤ If-0  ─»
              ░      └───┘ ░        ║   ║   └──╥───  └───┘ ───────┘   └──╥───  »
                                    ║   ║ ┌────╨────┐               ┌────╨────┐»
c: 3/═══════════════════════════════╩═══╩═╡ c_0=0x1 ╞═══════════════╡ c_1=0x1 ╞»
                                    1   0 └─────────┘               └─────────┘»
«                     ░            
«q_0: ────────────────░────────────
«                     ░            
«q_1: ────────────────░────────────
«     ┌───┐ ───────┐  ░ ┌──────┐┌─┐
«q_2: ┤ X ├  End-0 ├──░─┤ post ├┤M├
«     └───┘ ───────┘  ░ └──────┘└╥┘
«c: 3/═══════════════════════════╩═
«                                2

## Success Probabilities
Over all noise configs and probes, and whether Bob corrects
* If Bob corrects, P(success) is decently high
* If Bob does not, P(success) is basically 50/50
* P(success) decreases as more noise is added: Error(gates), Error(readout) < Error(combined)

In [8]:
for corrected in (True, False):
    print(f"Corrected: {corrected}")
    for noise in NOISE:
        res = {}
        for probe in PROBES:
            qt = quantum_teleportation(probe,  corrected)
            result, _ = run(qt, noise)
            result_dict = result.get_counts()
            res[probe] = round(marginal_counts(result_dict, [2])["0"] / SHOTS, 4)
        print(f"- {noise}: {res}")
    print(f"{'=' * 10}")

Corrected: True
- ideal: {'0': 1.0, '1': 1.0, '+': 1.0, '-': 1.0, '+i': 1.0, '-i': 1.0}
- gates: {'0': 0.9766, '1': 0.9719, '+': 0.9653, '-': 0.9595, '+i': 0.9595, '-i': 0.9595}
- readout: {'0': 0.9414, '1': 0.9414, '+': 0.9368, '-': 0.9368, '+i': 0.9099, '-i': 0.9099}
- combined: {'0': 0.9216, '1': 0.9172, '+': 0.9087, '-': 0.905, '+i': 0.8804, '-i': 0.8804}
Corrected: False
- ideal: {'0': 0.501, '1': 0.501, '+': 0.501, '-': 0.501, '+i': 0.501, '-i': 0.501}
- gates: {'0': 0.501, '1': 0.501, '+': 0.501, '-': 0.501, '+i': 0.501, '-i': 0.501}
- readout: {'0': 0.499, '1': 0.499, '+': 0.499, '-': 0.499, '+i': 0.499, '-i': 0.499}
- combined: {'0': 0.499, '1': 0.499, '+': 0.499, '-': 0.499, '+i': 0.499, '-i': 0.499}


## Questions
* Why the phase-sensitive probes are useful: To proof the generalisability of the teleportation to multiple bases (X, Y and Z), the probes must be phase-sensitive to show that the teleportation (in the noiseless case) is not only sound in terms of bits, but in correctly transmitting phase as well.

* Why Bob needs Alice’s outcomes: Alice applies a Hadamard measurement on her two qubits after introducing some entanglement with Bob's qubit, causing Bob's qubit to deviate from the original unknown qubit, meaning Bob has to correct this perturbation in order to accurately "receive" the qubit from the teleportation. Thus, Alice must supply Bob with her measurement.

* Why readout errors can affect corrections: Because Bob relies on Alice's readouts to correct, if the readout induces an error, it can cause Bob to correct his qubit wrongly. This may introduce a mismatch between the final state of Bob's qubit and the state of the original unknown qubit, meaning the teleportation fails.

# Part B: Superdense Coding

In [ ]:
TWO_QUBITS = ["00", "01", "10", "11"]

# get entangle gates 
def entangle():
    entangle = QuantumCircuit(2, name="entangle")
    entangle.h(0)
    entangle.cx(0, 1)
    return entangle

def superdense_coding(message):
    # q[0, 1] = e-bits, c[0, 1] = clbits
    q = QuantumCircuit(2, 2)
    q.append(entangle(), [0, 1])
    q.barrier()

    # alice performs ops depending on the message
    if message[0] == "1":
        q.z(0)
    if message[1] == "1":
        q.x(0)

    # alice sends to bob
    q.barrier()

    # bob performs bell measurement
    # q0 -> c1, q1 -> c0
    q.cx(0, 1)
    q.h(0)
    q.measure(0, 1) # flip the order since q0 determines phase flip
    q.measure(1, 0)
    return q

## Circuit
Assuming the message sent is "11" and entangled bit preparation is included

In [10]:
superdense_coding("11").draw()

┌───────────┐ ░ ┌───┐┌───┐ ░      ┌───┐┌─┐
q_0: ┤0          ├─░─┤ Z ├┤ X ├─░───■──┤ H ├┤M├
     │  entangle │ ░ └───┘└───┘ ░ ┌─┴─┐└┬─┬┘└╥┘
q_1: ┤1          ├─░────────────░─┤ X ├─┤M├──╫─
     └───────────┘ ░            ░ └───┘ └╥┘  ║ 
c: 2/════════════════════════════════════╩═══╩═
                                         0   1

## Success Probabilities
Across all noises and messages
* Same phenomena as Part A

In [11]:
for noise in NOISE:
    res = {}
    for message in TWO_QUBITS:
        qsc = superdense_coding(message)
        result, _ = run(qsc, noise)
        res[message] = result.get_counts()[message] / SHOTS
    print(f"- {noise}: {res}")

- ideal: {'00': 1.0, '01': 1.0, '10': 1.0, '11': 1.0}
- gates: {'00': 0.963134765625, '01': 0.956298828125, '10': 0.958984375, '11': 0.955078125}
- readout: {'00': 0.94189453125, '01': 0.939697265625, '10': 0.94580078125, '11': 0.944091796875}
- combined: {'00': 0.906005859375, '01': 0.89794921875, '10': 0.9072265625, '11': 0.90380859375}


## Repeating `readout`
* Std is tighter for higher shots but the expected value appears to be similar

In [12]:
_message = "10"
_noise = "readout"
_shots = [256, 8192]
_seeds = [7, 8, 9]

for shot in _shots:
    for seed in _seeds:
        qsc = superdense_coding(_message)
        result, tqc = run(qsc, _noise, shot, seed)
        p = result.get_counts()[_message] / shot
        std = np.sqrt(p * (1 - p) / shot)
        print(f"{shot} shots, seed = {seed}: {p} +- {std}")

256 shots, seed = 7: 0.9296875 +- 0.01597955876167341
256 shots, seed = 8: 0.91015625 +- 0.017872361101749213
256 shots, seed = 9: 0.94140625 +- 0.014678923355036537
8192 shots, seed = 7: 0.9427490234375 +- 0.002566814732644567
8192 shots, seed = 8: 0.9420166015625 +- 0.0025821777791048012
8192 shots, seed = 9: 0.9437255859375 +- 0.0025461464732675536


## Questions
* Why more shots do not remove the readout error: The readout error is independent of the number of shots, and the recorded P(success) in this case should approach P = (1 - r)^2 = 0.97^2 = 0.9409 since two classical bits are read on Bob's side

* Why we still need a preshared Bell pair + 1 qubit transmission: The Bell pair enables 4 possible orthogonal Bell states compared to a single qubit (which by the Holevo bound only can store 1 bit of information), and the 1 qubit transmission needs to happen because Bob's qubit is not directly affected by Alice's (i.e. his is still maximally mixed), meaning Bob will need both qubits (and the information encoded in their entanglement) to extract the information of the 2 classical bits.

# Part C: Deutsch–Jozsa

In [43]:
ORACLE_KINDS = ["f=0", "f=1", "f=x0", "f=x1XORx0"]

def oracle(oracle_kind):
    o = QuantumCircuit(3, name="oracle")

    # if f = 0, do nothing since y XOR 0 = y
    if oracle_kind == "f=0":
        pass

    # if f = 1, always flip q2 since y XOR 1 = NOT y
    elif oracle_kind == "f=1":
        o.x(2)

    # if f = x0, flip q2 if q0 = 1, else do nothing (equiv CNOT)
    elif oracle_kind == "f=x0":
        o.cx(0, 2)

    # if f = x1 XOR x0, y XOR x1 XOR x0 is equivalent to x0 XOR (x1 XOR y)
    # so flip y if x1 or x2 are 1, or both
    elif oracle_kind == "f=x1XORx0":
        o.cx(0, 2)
        o.cx(1, 2)

    return o

def deutsch_jozsa(oracle_kind):
    # q[0..2] = e-bits, q[3] = clbits to measure answer
    q = QuantumCircuit(3, 2)

    # prepare 2nd qubit to ket(-)
    q.x(2)
    q.h(2)
    q.barrier()

    # place q0 and q1 in superposition
    q.h([0, 1])

    # ask oracle
    q.append(oracle(oracle_kind), [0, 1, 2])

    # measure q0 and q1
    q.h([0, 1])
    q.measure(0, 0)
    q.measure(1, 1)
    return q

## Circuit

In [44]:
deutsch_jozsa("f=0").draw()

░ ┌───┐┌─────────┐┌───┐┌─┐   
q_0: ───────────░─┤ H ├┤0        ├┤ H ├┤M├───
                ░ ├───┤│         │├───┤└╥┘┌─┐
q_1: ───────────░─┤ H ├┤1 oracle ├┤ H ├─╫─┤M├
     ┌───┐┌───┐ ░ └───┘│         │└───┘ ║ └╥┘
q_2: ┤ X ├┤ H ├─░──────┤2        ├──────╫──╫─
     └───┘└───┘ ░      └─────────┘      ║  ║ 
c: 2/═══════════════════════════════════╩══╩═
                                        0  1

## Success Probabilities
* Again, P(success) decrease as the errors increase in severity: P_success(gates) and P_success(readout) < P_success(combined)

In [15]:
for noise in NOISE:
    res = {}
    for oracle_type in ORACLE_KINDS:
        qdj = deutsch_jozsa(oracle_type)
        result, _ = run(qdj, noise)

        # if balanced, expect "00" as output
        if oracle_type in ("f=0", "f=1"):
            res[oracle_type] = result.get_counts()["00"] / SHOTS

        # else, 01, 10, 11 are expected. need sum
        else:
            summation = 0
            r = result.get_counts()
            for k in r:
                if k != "00":
                    summation += r[k]
            res[oracle_type] = summation / SHOTS
    print(f"- {noise}: {res}")

- ideal: {'f=0': 1.0, 'f=1': 1.0, 'f=x0': 1.0, 'f=x1XORx0': 1.0}
- gates: {'f=0': 0.989990234375, 'f=1': 0.989990234375, 'f=x0': 0.978515625, 'f=x1XORx0': 0.988037109375}
- readout: {'f=0': 0.94189453125, 'f=1': 0.94189453125, 'f=x0': 0.969482421875, 'f=x1XORx0': 0.998291015625}
- combined: {'f=0': 0.932861328125, 'f=1': 0.932861328125, 'f=x0': 0.949951171875, 'f=x1XORx0': 0.9853515625}


## Questions
* Role of phase kickback: Because q0 and q1 are in the ket(+) state (due to the H gates at the start), if the function is balanced, then the CNOT on q2 depends on if f=x0 or f=x1 XOR x0. This means that for either case, q0 (and q1 for the latter case) will "pick up" the negative phase from q2 since ket(-) is an eigenket of X and will flip the phase for the ket(1) in ket(+), transforming it to ket(-).

* Role of the final H gates: Once the above kickback happens, the H gates will prepare q0 and q1 to be measured in the computational basis of Z. If the function is balanced, then at least one qubit in q0 or q1 will be ket(1), otherwise both will be ket(0).

* Why is a noisy nonzero outcome no longer a conclusive test: If noise is introduced, then it is possible for readout or gate errors to perturbate the results such that a balanced function can appear to be constant and vice versa. One result is thus not sufficiently representative or conclusive of the nature of the unknown function and oracle

# Part D: Grover Search

In [ ]:
def oracle(q):
    # mark ket(10)
    # so ket(10) -> X(1) -> ket(11) -> CZ(0, 1) -> -ket(11)
    q.x(1)
    q.cz(0, 1)
    q.x(1)

def diffuse(q):
    q.h([0, 1])
    q.x([0, 1])
    q.cz(0, 1)
    q.x([0, 1])
    q.h([0, 1])
    return q

def grover_search(itera=1, redundant=0):
    # assume ket(10) means q0 = ket(1), q1 = ket(0)
    q = QuantumCircuit(2, 2)
    q.h([0, 1])
    q.barrier()

    # add redundant CNOT pairs if need
    for _ in range(redundant):
        q.cx(0, 1)
        q.cx(0, 1)
    q.barrier()

    # grover iters
    for _ in range(itera):
        oracle(q)
        diffuse(q)
        q.barrier()

    # reverse the order due to the mapping
    q.measure(0, 1)
    q.measure(1, 0)
    return q

## Circuit
For a Grover Search with 1 pair of unnecessary gates and 1 iteration

In [34]:
grover_search(1, 1).draw()

┌───┐ ░            ░         ┌───┐┌───┐        ┌───┐┌───┐ ░ ┌─┐   
q_0: ┤ H ├─░───■────■───░───────■─┤ H ├┤ X ├──────■─┤ X ├┤ H ├─░─┤M├───
     ├───┤ ░ ┌─┴─┐┌─┴─┐ ░ ┌───┐ │ ├───┤├───┤┌───┐ │ ├───┤├───┤ ░ └╥┘┌─┐
q_1: ┤ H ├─░─┤ X ├┤ X ├─░─┤ X ├─■─┤ X ├┤ H ├┤ X ├─■─┤ X ├┤ H ├─░──╫─┤M├
     └───┘ ░ └───┘└───┘ ░ └───┘   └───┘└───┘└───┘   └───┘└───┘ ░  ║ └╥┘
c: 2/═════════════════════════════════════════════════════════════╩══╩═
                                                                  1  0

## Success Probabilities

In [36]:
ITERS = [0, 1, 2]
for noise in NOISE:
    res = {}
    for iters in ITERS:
        qgs = grover_search(iters)
        result, _ = run(qgs, noise)
        res[iters] = result.get_counts().get("10", 0) / SHOTS
    print(f"{noise}: {res}")

ideal: {0: 0.24560546875, 1: 1.0, 2: 0.24560546875}
gates: {0: 0.24560546875, 1: 0.917236328125, 2: 0.24560546875}
readout: {0: 0.243408203125, 1: 0.94580078125, 2: 0.243408203125}
combined: {0: 0.243408203125, 1: 0.86865234375, 2: 0.243408203125}


## Unnecessary Gates

In [38]:
_iters = 1
_redundants = [0, 5]
_noise = ["ideal", "combined"]

for noise in _noise:
    res = {}
    for red in _redundants:
        qgs = grover_search(_iters, red)
        result, _ = run(qgs, noise)
        res[red] = result.get_counts().get("10", 0) / SHOTS
    print(f"{noise}: {res}")

ideal: {0: 1.0, 5: 1.0}
combined: {0: 0.86865234375, 5: 0.762451171875}
